# Python Crime Data Engineering Pipeline Project

### Introduction 


project introduction

### Workflow Diagram 

*insert data diagram

### Data Assumptions

• Data assumption 1...  
• Data assumption 2...

### Import Python Libraries

In [3]:
#data handling libraries 
import pandas as pd 
import numpy as np

#file and path management 
from pathlib import Path 
import os 
import glob

### Ingestion Layer

**Outline:** 

This layer follows the steps taken to load data into the notebook for later cleaning and transformation. 

**1a. Load Crime Data in batches, by police force**

The data is loaded in batches to avoid loading the full dataset into memory at once. This ensures a more scaleable pipeline and an easier debugging by police force process.  

For example, when an error occurs, Python can specify which batch(police force) caused this error, narrowing down the number of files to check. 

**1b. Validate file upload**

The number of rows and files loaded will be printed. This allows for loading validation and comparison between raw and processed data. 

In [4]:
#define path to raw crime data folder 
raw_data = Path("data")/ "raw_data" / "crime_data" #create path object 

#create a list of the forces to loop through
forces = ['metropolitan','thames-valley','west-midlands','west-yorkshire']

#create an empty list for each batch of forces to be added to 
batches = [] 

#create a for loop that locates individual police force files and adds them to the batches list 
for force in forces: 
    #create a list of csv files to loop through 
    csv_files = list(raw_data.glob(f"**/*{force}*street.csv")) #** - glob searches in sub-folders, * - wildcard spot 

    #create an empty list for each file per force to be added to, resets every outer loop iteration  
    force_batch = []
    #create a nested for loop that reads each police force file in batches 
    for file in csv_files: 
        df = pd.read_csv(file) #create a dataframe for each file 
        force_batch.append(df) #add each dataframe to force_batch list 

    #create a dataframe with all files from each force 
    force_df = pd.concat(force_batch, ignore_index=True) #ignore_index=True - resets index + prevents duplicates in force_df 
    print(f"\n{force}: {len(force_df)} rows loaded from {len(csv_files)} files\n") #validation check 
    batches.append(force_df) #add each force batch to batches list 

#once all files have been retrieved from each force, combine all files into one raw dataframe 
crime_raw = pd.concat(batches, ignore_index=True)
print(f"\nTotal rows loaded: {len(crime_raw)}\n")


metropolitan: 2386012 rows loaded from 25 files


thames-valley: 413128 rows loaded from 25 files


west-midlands: 712238 rows loaded from 25 files


west-yorkshire: 647272 rows loaded from 25 files


Total rows loaded: 4158650



**2. Load Enrichment Data**

The load_file function is created as a standardised step to reduce repetition in the code. For example, the file type check and print statement only need to be written once inside the function. When calling the function, arguments will be specified individually to account for the varying file structures. 

In [5]:
# create a function to load excel/csv files

def load_file(filepath, sheet_name=None, header=0): 
    """ 
    loads a CSV or Excel file based on the file extension. 
    
    Parameters: 
        filepath = path to the file (defined when the function is called)  
        sheet_name = Excel sheet name (only for Excel files)
        header = row to use as column headers - 0 indexed (only for Excel files)
        
    Returns: 
    df = dataframe to assign to a variable 
    """
    filepath = Path(filepath) #converts first argument to a path object 

    if filepath.suffix == '.csv': #checks file suffix (pathlib attribute)
        df = pd.read_csv(filepath)
    elif filepath.suffix in ['.xlsx','.xls']: #accounts for multiple Excel file suffix'
        df = pd.read_excel(filepath, 
                          sheet_name = sheet_name, 
                           header = header 
                          ) #assigns what ever is passed when the function is called 
    else: 
        raise ValueError(f"Unsupported file type: {filepath.suffix}") #clearly define error 

    print (f"{filepath.name}: {len(df)} rows loaded.") #prints full file name + suffix (pathlib attribute)
    return df 
        
#define path to raw data folder 
raw_data = Path("data")/ "raw_data"

**2a. Load Population enrichment data**

In [6]:
pop_raw = load_file(raw_data/ "policeforceareas1991to2024.xlsx",
                        sheet_name="Mid-2021 to Mid-2024", 
                        header=3) #assigns 4th row (0 indexed)as column headers 

policeforceareas1991to2024.xlsx: 172 rows loaded.


**2b. Load Deprivation enrichment data**

In [7]:
dep_raw = load_file(raw_data/"deprivation_data.csv")

deprivation_data.csv: 33755 rows loaded.


**2c. Load Income & Housing enrichment data**

In [8]:
#load median house price sheet 
hos_raw = load_file(raw_data/ "aff1ratioofhousepricetoworkplacebasedearnings.xlsx",
                        sheet_name="5a", 
                        header=1) #assigns 2nd row (0 indexed)as column headers 

#load median workplace-based earnings sheet 
inc_raw = load_file(raw_data/ "aff1ratioofhousepricetoworkplacebasedearnings.xlsx",
                        sheet_name="5b", 
                        header=1) #assigns 2nd row (0 indexed)as column headers 

aff1ratioofhousepricetoworkplacebasedearnings.xlsx: 318 rows loaded.
aff1ratioofhousepricetoworkplacebasedearnings.xlsx: 318 rows loaded.


### Cleaning & Validation Layer


**Outline:** 

This layer follows the steps taken to clean the data, including data quality assessments.

**1. Data Quality Assessment**

The qual_check function is created as a standardised step to observe initial dataset impressions including column names, row & column count, datatypes, nulls, etc., reducing repitition in the code. When calling the function, arguments will be specified individually to account for the varying dataset structures. 

In [9]:
#create a function to check for data quality issues 

def qual_check(df, name, cat_cols=None):
    """ 
    perfoms a data quality check on a given dataframe. 

    Parameters: 
    df = given dataframe to assess
    name = variable name given to the dataframe (used in print statements)
    cat_cols = list of categorical columns to check for format incosistencies/unique values  
    """
    print(f"Data Quality Check: {name}")

    #dataset preview 
    print("\nFirst 3 Rows:\n")
    display(df.head(3))

    #dataset shape 
    print(f"\nNumber of Rows and Columns: {df.shape}")

    #column data types 
    print(f"\nColumn Data Types:\n\n{df.dtypes}")

    #null summary 
    null_counts = df.isnull().sum()
    null_pct = (df.isnull().sum() / len(df) * 100).round(0)
    null_summary = pd.DataFrame({ #create null summary dataframe 
    'Null Count': null_counts, 
    'Null %': null_pct
    })
    print("\nColumns with null values:\n")
    display(null_summary[null_summary['Null Count'] > 0]) #only shows columns with null values

    #duplicate check 
    print(f"\nRow Duplicates Count: {df.duplicated().sum()}")

    #categorical columns unique values check 
    if cat_cols: 
        print("\nCategorical Columns Unique Values:\n")
        for col in cat_cols: 
            if col in df.columns: 
                print(f"\nColumn Name: {col}\n")
                print(f"\nTotal unique values: {df[col].nunique()}\n")
                print(f"\nUnique values:\n{df[col].unique()}") 
            else: 
                raise ValueError(f"Column '{col}' not found in '{name}'"
                                f"Available columns are: {df.columns.tolist()}")           

**1a. Crime Data**

Data Quality Issues: 

***Null values***  
    • Context = 100% null  
    • Crime ID = 16% null  
    • Last outcome category = 16% null  
    • Longitude = 1% null  
    • Latitude = 1% null  
    • LSOA code = 1% null  
    • LSOA name = 1% null   

Where LSOA code is null, LSOA name, Longitude, and Latitude is also null. The null LSOA code and LSOA name values limits merge ability with dep_raw, resulting in a loss of deprivation data. See step x for data cleaning process.

***Multi-Value attributes***     
    • Month - stores month and year info   

This multi-value attribute will impact the final reporting grain. See step x for data cleaning process. 

***Duplicate Records***      
    • 262,672 duplicate records in the dataset  

Duplicate records may skew analysis. See step x for data cleaning process.
    

In [ ]:
qual_check(crime_raw, "Raw Crime Data", cat_cols=['Reported by', 'Falls within', 'LSOA name', 'Crime type', 'Last outcome category'])

In [ ]:
display(crime_raw[crime_raw['LSOA code'].isnull()])

**1b. Population Data**

Data Quality Issues: 

***Inconsistent Format*** of police force names between pop_raw & crime_raw  

• Metropolitan Police vs Metropolitan Police Service  
• Thames Valley vs Thames Valley Police  
• West Midlands vs West Midlands Police     
• West Yorkshire vs West Yorkshire Police  

These inconsistencies may cause issues when merging the pop_raw and crime_raw datasets. See step x for data cleaning process.

In [ ]:
qual_check(pop_raw, "Raw Population Data", cat_cols=['PFA 2023 Name'])

**1c. Deprivation Data**

Data Quality Issues: 

***Long Column Names***

May reduce readability of final dataset and dashboard. See step x for data cleaning process.

In [ ]:
qual_check(dep_raw, "Raw Deprivation Data", cat_cols=['LSOA name (2021)', 'Local Authority District name (2024)'])

**1d. Housing Data**

Data Quality Issues: 

***Incompatible Grains***

Each record in the dataset has a median house price for each year (each year has it's own column). Unlike the crime_raw dataset, where each record refers to a specified month/year, only storing data relevant to that month/year. This may cause issues when merging the datasets. See step x for the data cleaning process. 

***Inconsistent Format*** of year columns' names between hos_raw and inc_raw

These inconsistencies may cause issues when grouping by year. See step x for data cleaning process.

In [ ]:
qual_check(hos_raw, "Raw Housing Data", cat_cols=['Country/Region name', 'Local authority name'])

**1e. Income Data**

Data Quality Issues: 

***Incompatible Grains***

Each record in the dataset has a median income for each year (each year has it's own column). Unlike the crime_raw dataset, where each record refers to a specified month/year, only storing data relevant to that month/year. This may cause issues when merging the datasets. See step x for the data cleaning process. 

***Income data stored as object***

This may cause errors or unexpected results during mathematical operations and won't be recognised as a measure in BI tools. See step x for the data cleaning process.

In [ ]:
qual_check(inc_raw, "Raw Income Data", cat_cols=['Country/Region name', 'Local authority name'])

**2. Clean Data**

The clean_data function is created as a standardised step to clean the datasets based on the discovered data quality issues. When calling the function, the arguments...

Data Cleaning Steps: 

1. Create a copy of the raw data to allow access and comparison to the original data
2. Remove duplicates rows before other data cleaning steps to reduce processing time by avoiding unnecassary cleaning of rows 
3. Drop columns with more than 50% null values before other data cleaning steps to reduce processing time by avoiding filling in null values that will be dropped
4. Fill null values according to data type to prevent later aggregation errors
5. Split crime_raw's 'Month' column into 'Year' and 'Month' columns to enable time-based aggregations and filtering at both monthly and annual levels
6. Standardise police force naming convention allow joins between the crime_raw and pop_raw data

In [179]:
#create a function that corrects the data quality issues found

def clean_data(df, df_name, drop_cols=None,force_name_dict=None, force_col=None, year_cols=None, year_dict=None, value_name=None): 
    """
    performs data cleaning tasks on a given dataframe. 

    Parameters: 
    df = given dataframe to clean
    df_name = descriptive name given to df and used in print statements 
    drop_cols = list of columns wanting to drop 
    force_name_dict = dictionary storing standardised police force names 
    force_col = df column containing police force names 
    year_cols = list of year columns 
    year_dict = dictionary storing standardised years 
    value_name = name given to values column after pivoting 

    Returns:
    clean_df = cleaned dataframe 
    """
    clean_df = df.copy()
    rows_in = len(clean_df) 
    print(f"\nNumber of rows in {df_name} before cleaning: {rows_in}\n")

    #remove duplicate rows 
    dupes = clean_df.duplicated().sum()

    print(f"\nChecking for duplicate rows in {df_name}")
    
    if dupes > 0: 
        
        print(f"\nDropping {dupes} duplicate rows")
        
        clean_df = clean_df.drop_duplicates().reset_index(drop=True)
        
        print (f"\nNumber of rows after dropping duplicates: {len(clean_df)}\n")
    else: 
        
        print(f"\nNo duplicates rows found in {df_name}\n")

    #drop unwanted columns 
    print(f"\nChecking for columns to drop from {df_name}")
    
    if drop_cols: 

        print(f"\nDropping columns: {drop_cols}")

        clean_df = clean_df.drop(columns=drop_cols)  

        print(f"\nNumber of columns dropped: {len(drop_cols)}")
        print(f"\nNumber of columns left: {clean_df.shape[1]}\n")
    else: 

        print(f"\nNo columns to drop from {df_name}\n")
    
    #drop columns with > 50% nulls 
    null_pct = (clean_df.isnull().sum() / len(clean_df) * 100).round(0) #creates a pandas series (list of dataframe columns)
    null_cols = null_pct[null_pct > 50].index.tolist() #returns row lables of columns with > 50% nulls and adds them to a list 

    print(f"\nChecking for columns with >50% null values in {df_name}")
    
    if null_cols: #checks if list contains any values 
        
        print(f"\nDropping columns: {null_cols}")
        
        clean_df = clean_df.drop(columns=null_cols)  

        print(f"\nNumber of columns dropped: {len(null_cols)}\n")
        print(f"\nNumber of columns after dropping nulls: {clean_df.shape[1]}\n")
    else: 

        print(f"\nNo columns with >50% null values found in {df_name}\n")

    #fill other null values with column mean values by police force or "Unknown"
    for col in clean_df.columns: #loops through a list of dataframe columns 

        print(f"\nChecking for null values in {col}")
        null_count = clean_df[col].isnull().sum()
        
        if null_count > 0: 
            
            print(f"\nFilling {null_count} null values in '{col}'") 
        
            if pd.api.types.is_integer_dtype(clean_df[col]) or pd.api.types.is_float_dtype(clean_df[col]):
                
               #calculate mean per force and map back to each row 
               col_mean_by_force = clean_df.groupby(force_col)[col].transform('mean')
               clean_df[col] = clean_df[col].fillna(col_mean_by_force) 
                
               print(f"\nNull values filled with force-level average in {col}")

            else: #treats dataframe column values as objects/strings if they are not numerical 
                clean_df[col] = clean_df[col].fillna("Unknown") #fills nulls with "Unknown" in clean_df

                print(f"\nNull values filled with 'Unknown' in {col}")
               
            print(f"\nNumber of null values filled: {null_count}\n")
        else: 

            print(f"\nNo null values found in {col}\n")

    #split month column into year and month 
    if 'Month' in clean_df.columns: 
        
        print("\nSplitting 'Month' column into 'Year' and 'Month'")
        
        dates = pd.to_datetime(clean_df['Month'], format="%Y-%m") #creates a series (list of column values)

        month_index = clean_df.columns.get_loc("Month") #finds location of Month column 

        clean_df.insert(month_index + 1, "Year", dates.dt.year) #inserts a new Year column, defining it's position, name, and values (extracted years)
        
        clean_df['Month'] = dates.dt.month #overwrites month column with extracted months 
        
        print("\n'Month' column split into 'Year' and 'Month'\n")

    #Standardise police force names 
    if force_name_dict: #checks if force_name_dict passed in function, otherwise ignored 
        
        if force_col in clean_df.columns: #checks if force_col exists in the dataframe, otherwise returns an error 
            
            print(f"\nStandardising police force names in column: '{force_col}'")
            
            clean_df[force_col] = clean_df[force_col].replace(force_name_dict) #replaces each instance of the force_name_dict key with corresponding value
            
            print(f"\nPolice force names standardised in column: '{force_col}'\n")
        else: 
            raise ValueError(f"\nColumn '{force_col}' not found in {df_name}\n") 

    #unpivot year columns into rows 
    if year_cols: #checks if year_cols is passed in the function, otherwise ignored 
        
        print("\nUnpivoting Year columns into rows")
        
        id_cols = [col for col in clean_df.columns if col not in year_cols] #a list of columns that are NOT year columns, used as identifier columns to stay fixed during the unpivot
        
        clean_df = clean_df.melt( #pandas method that converts columns into rows 
            id_vars=id_cols, #keep non-year columns as fixed identifiers 
            value_vars=year_cols, #unpivot year columns into rows 
            var_name='Year', #names column containing year values 
            value_name=value_name #names new column storing values from original year columns 
        )
        
        print("\nYear columns unpivoted to rows")
        print(f"\nValue column created: '{value_name}'\n")
        
    #standardise year columns 
    if year_dict: #checks if year_dict is passed in function, otherwise ignores 

        if 'Year' in clean_df.columns: #checks if Year exists in the dataframe, otherwise returns an error 
            
            print("\nStandardising values in 'Year' column")
            
            clean_df['Year'] = clean_df['Year'].replace(year_dict) #replaces each instance of the year_dict key with corresponding value
            
            print("\nValues standardised in 'Year' column\n")
        else: 
            raise ValueError(f"\nColumn 'Year' not found in {df_name}\n") 

    #convert numeric values stored as objects 
    print("\nAttempting to convert object columns to numerical data types") 
    
    for col in clean_df.select_dtypes(include='object').columns: #creates a list of columns storing object values 
        try: 
            clean_df[col] = clean_df[col].replace('[x]', np.nan)
            clean_df[col] = pd.to_numeric(clean_df[col], errors='raise') #attempts to convert to numerical, raises error if not possible & exuctes except statement 
            print(f"\nNumeric objects converted to numeric data type in column: '{col}'") #prints if conversion is successful 
        except: 
            pass #moves on to next column if an error occurs 
            
    print("\nNumeric conversions attempted on all object columns\n")

    #cleaning validation check 
    rows_out = len(clean_df)
    rows_dropped = rows_in - rows_out 

    print(f"\n{'-'*40}") #line separator between cleaning and validation check 
    print(f"\nNumber of rows in {df_name} before cleaning: {rows_in}")
    print(f"\nNumber of rows in {df_name} after cleaning: {rows_out}")
    print(f"\nNumber of rows dropped from {df_name} during cleaning: {rows_dropped}")
    print("\nCleaning complete!\n")

    return clean_df
        

**2a. Crime Data**

Data cleaning steps performed: 

1. Removed duplicate rows
2. Remove unwanted columns  
   • ***Crime ID*** - individual crime records are not needed within the reporting grain and will not exist after aggregation  
   • ***Reported by*** - redundant
3. Remove Context column - 100% null values
4. Fill null values as column value mean by police force or Unknown - measures vary by police force
5. Split Month column into Month and Year columns
6. Standardise police force names 

In [14]:
#create list that will enable the clean_data function to drop unwanted columns 
drop_cols = ['Crime ID','Reported by']

In [15]:
#create force name dictionary that will allow for standardisation of police force names
force_name_dict = {
    'Metropolitan Police Service': 'Metropolitan Police', 
    'Thames Valley': 'Thames Valley Police', 
    'West Midlands ': 'West Midlands Police', 
    'West Yorkshire': 'West Yorkshire Police'
}

In [23]:
#run function + reassign cleaned dataframe to a new variable 
crime_clean = clean_data(crime_raw, 'Crime Data', drop_cols=drop_cols, force_name_dict=force_name_dict, force_col='Falls within')

In [25]:
#perform data quality check on cleaned data
qual_check(crime_clean, "Cleaned Crime Data", cat_cols=['Falls within', 'LSOA name', 'Crime type', 'Last outcome category'])

**2b. Population Data**

Data cleaning steps performed: 

1. Aggregate population columns - avoid unnessacarily wide dataframe, especially when merging
2. Add rows with estimated 2025 population values based on England population growth in 2024 (1.2%) (Office for National Statistics,2025) - prevent loss of 2025 when merged with crime dataframe
3. Standardise police force names

In [180]:
# aggregate population columns to a single column 
cat_cols = ['PFA 2023 Code','PFA 2023 Name'] #categorical columns 
pop_cols = [col for col in pop_raw.columns if col not in cat_cols] 

#sum population columns vertically (by values in each column) and horizontally (across columns)
pop_pre = pop_raw.groupby(['PFA 2023 Name', 'Year'])[pop_cols].sum().sum(axis=1).reset_index()
pop_pre.columns = ['Force', 'Year', 'Population']

pop_pre.head(4)

In [27]:
def proj_pop(df, force_col, year_col, pop_col, proj_year): 
    """
    calculates projected 2025 population values for each police force 

    Parameters:
    df = population dataframe 
    force_col = column containing police force names 
    year_col = column containing year values 
    pop_col = column containing population values 
    proj_year = year to project population value for 

    Returns: 
    df = population dataframe with projected years appended 
    """
    proj_rows = [] #creates an updateable list of projected rows to add the the population dataframe 

    for force in df[force_col].unique(): #loop through the list of force column unique values 
        
        #creates a filtered dataframe with current force in loop 
        force_df = df[df[force_col] == force].sort_values(year_col, ascending=True)
        
        #find the most recent year's population data 
        latest_year = force_df[year_col].iloc[-1] #searches for latest year in force_df year column 
        latest_pop = force_df[pop_col].iloc[-1]  #searches for latest population value in force_df population column

        proj_pop = round(latest_pop * 1.012) #calculates a projected 1.2% population increase 

        #adds projected population rows to proj_rows list
        proj_rows.append({
           force_col: force, 
           year_col: proj_year, 
           pop_col: proj_pop    
        })

    #convert projected rows list to a dataframe  
    proj_df = pd.DataFrame(proj_rows)

    #concating population dataframe with projected rows dataframe 
    df = pd.concat([df,proj_df], ignore_index=True).sort_values(
        [force_col,year_col], ascending=True).reset_index(drop=True) #sort by police force and year 

    print(f"\n{proj_year} population projections appended for {len(proj_rows)} forces\n")

    return df 

In [28]:
pop_pre2 = proj_pop(pop_pre, 'Force', 'Year', 'Population', 2025)
pop_pre2.head(5)


2025 population projections appended for 43 forces



,Force,Year,Population
0,Avon and Somerset,2021,1747096
1,Avon and Somerset,2022,1768932
2,Avon and Somerset,2023,1791987
3,Avon and Somerset,2024,1815689
4,Avon and Somerset,2025,1837477


In [29]:
#create force name dictionary that will allow for standardisation of police force names
force_name_dict = {
    'Metropolitan Police': 'Metropolitan Police', 
    'Thames Valley': 'Thames Valley Police', 
    'West Midlands': 'West Midlands Police', 
    'West Yorkshire': 'West Yorkshire Police'
}

In [31]:
#run function + reassign cleaned dataframe to a new variable 
pop_clean = clean_data(pop_pre2, 'Population Data',force_name_dict=force_name_dict, force_col='Force')

In [33]:
#perform data quality check on cleaned data
qual_check(pop_clean, "Cleaned Population Data", cat_cols=['Force'])

**2c. Deprivation Data**

Data cleaning steps performed: 

1. Dropping unwanted columns/kept the following:    
   • LSOA code (2021) - join key with crime data  
   • Local Authority District code (2024) - join key with housing & income data  
   • Local Authority District name (2024) - descriptive name for code
   • Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs) - easier interpretation of deprivation  
   • + Other deprivation dimensions (domain) scores columns - allow for force level aggregation    

In [34]:
#columns wish to keep 
keep_cols = [
    'LSOA code (2021)',
    'Local Authority District code (2024)', 
    'Local Authority District name (2024)', 
    'Index of Multiple Deprivation (IMD) Score', 
    'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)',
    'Income Score (rate)',
    'Employment Score (rate)',
    'Education, Skills and Training Score',
    'Health Deprivation and Disability Score',
    'Crime Score',
    'Barriers to Housing and Services Score',
    'Living Environment Score'
            ]

In [35]:
drop_cols = [col for col in dep_raw.columns if col not in keep_cols]

In [37]:
#run function + reassign cleaned dataframe to a new variable 
dep_clean = clean_data(dep_raw, 'Deprivation Data', drop_cols=drop_cols)

In [39]:
#perform data quality check on cleaned data
qual_check(dep_clean, "Cleaned Deprivation Data", cat_cols=['Local Authority District name (2024)'])

**2d. Housing Data**

1. Drop unwanted columns  
   • Local authority name - redundant, present in deprivation data
2. Unpivot Year columns into rows

In [40]:
#list of columns passed in function for pivoting 
year_cols = ['Year ending Sep 2023','Year ending Sep 2024','Year ending Sep 2025']

In [41]:
#columns wish to keep
keep_cols = [*year_cols, 'Country/Region name', 'Local authority code']

In [42]:
#create list that will enable the clean_data function to drop unwanted columns 
drop_cols = [col for col in hos_raw.columns if col not in keep_cols]

In [43]:
#create year dictionary that will allow for standardisation of year values
year_dict = {
    'Year ending Sep 2023': 2023, 
    'Year ending Sep 2024': 2024, 
    'Year ending Sep 2025': 2025
}

In [46]:
#run function + reassign cleaned dataframe to a new variable 
hos_clean = clean_data(
    hos_raw, 
    'Housing Data', 
    drop_cols=drop_cols, 
    year_cols=year_cols, 
    year_dict=year_dict, 
    value_name= 'Median house price'
)

In [47]:
#perform data quality check on cleaned data
qual_check(hos_clean, "Cleaned Housing Data", cat_cols=['Country/Region name'])

**2e. Income Data**

1. Drop unwanted columns
2. Unpivot Year columns into rows
3. Convert income values to integers

In [181]:
#list of columns passed in function for pivoting
year_cols = ['2023','2024','2025']

In [182]:
#columns wish to keep
keep_cols = [*year_cols, 'Country/Region name', 'Local authority code']

In [183]:
#create list that will enable the clean_data function to drop unwanted columns 
drop_cols = [col for col in inc_raw.columns if col not in keep_cols]

In [184]:
#create year dictionary that will allow for standardisation of year values
year_dict = {
    '2023': 2023, 
    '2024': 2024, 
    '2025': 2025
}

In [197]:
#run function + reassign cleaned dataframe to a new variable 
inc_clean = clean_data(
    inc_raw, 
    'Income Data', 
    drop_cols=drop_cols, 
    year_cols=year_cols, 
    year_dict=year_dict, 
    value_name= 'Median workplace earnings'
)

In [198]:
#perform data quality check on cleaned data
qual_check(inc_clean, "Cleaned Income Data", cat_cols=['Country/Region name'])

### Feature Engineering & Transformation Layer

**Outline:** 

This layer follows the steps taken to transform the data, including joining dataframes and handling temporal grain mismatches.

**1. Merge Dataframes**

All dataframes will be merged together to allow for aggregations to the intended reporting grain. The merge_data function is a standardised process to merge the dataframes together. When calling the function, the arguments should be specified to account for varying dataframe structures.

In [83]:
def merge_data(left_df, left_df_name, right_df, right_df_name, merge_type, left_key, right_key, force_col=None): 
    """
    merges given dataframes. 

    Parameters: 
    left_df = left dataframe 
    left_df_name = descriptive name for left dataframe, used in print statements 
    right_df = right dataframe 
    right_df_name = descriptive name for right dataframe, used in print statements 
    merge_type = the type of merged intended to be performed 
    left_key = the name of the join key in the left table 
    right_key = the name of the join key in the right table 
    force_col = column containing force names

    Returns:
    merged_df = merged dataframe   
    """
    #states merging activity 
    print(f"Merging left dataframe: {left_df_name} with right dataframe: {right_df_name}\n") 
    print(f"Merge type: {merge_type} | Left key:'{left_key}'| Right key:'{right_key}'\n")

    #prints row count before merging 
    print(f"\nRows in {left_df_name} before merge: {len(left_df)}")
    print(f"\nRows in {right_df_name} before merge: {len(right_df)}")
        
    merged_df = pd.merge(
            left_df, 
            right_df, 
            how=merge_type, 
            left_on=left_key, 
            right_on=right_key
        )

    if merge_type != 'inner': #joins other than inner may return nulls

        if force_col: 

            for col in merged_df.columns: #loops through a list of dataframe columns 
    
                print(f"\nChecking for null values in {col}")
                null_count = merged_df[col].isnull().sum()
            
                if null_count > 0: 
                
                    print(f"\nFilling {null_count} null values in '{col}'") 
            
                    if pd.api.types.is_integer_dtype(merged_df[col]) or pd.api.types.is_float_dtype(merged_df[col]):
                    
                       #calculate mean per force and map back to each row 
                       col_mean_by_force = merged_df.groupby(force_col)[col].transform('mean')
                       merged_df[col] = merged_df[col].fillna(col_mean_by_force) 
                        
                       print(f"\nNull values filled with force-level average in {col}")
        
                    else: #treats dataframe column values as objects/strings if they are not numerical 
                        merged_df[col] = merged_df[col].fillna("Unknown") #fills nulls with "Unknown" in clean_df
        
                        print(f"\nNull values filled with 'Unknown' in {col}")
                       
                    print(f"\nNumber of null values filled: {null_count}\n")
                else: 
        
                    print(f"\nNo null values found in {col}\n")
            
    print(f"\n{left_df_name} merged with {right_df_name}\n")
    print(f"\nNumber of rows in merged dataframe: {len(merged_df)}\n")

    with pd.option_context('display.max_columns', None):
        display(merged_df.head(5))
        
    return merged_df 

**1a. Merge crime_clean & pop_clean**

The merge uses both year and police force columns as join keys to ensure a many-to-one relationship, avoiding many-to-many relationships. e.g. Many 'Metropolitan Police' x 2024 records in the crime dataframe can refer to one 'Metropolitan Police' x 2024 record in the population dataframe.  

In [ ]:
#check columns in crime_clean & pop_clean for join keys
display(crime_clean.head(3),pop_clean)

In [85]:
#run merge_data function 
crime_pop_merge = merge_data(
    crime_clean, 'Crime Data', 
    pop_clean, 'Population Data', 
    merge_type='inner', 
    left_key=['Falls within','Year'], 
    right_key=['Force','Year']
)

**1b. Merge crime_pop_merge & dep_clean**

The merge uses LSOA code columns as join keys creating a many-to-one relationship. e.g. Many crime_pop_clean records can refer to one record in the dep_clean dataframe.

In [71]:
#check for join keys 
display(crime_pop_merge.head(3), dep_clean.head(3))

In [63]:
#check for unique LSOA codes for each record, numbers should be the same if true 
dep_clean['LSOA code (2021)'].nunique() 
len(dep_clean)

33755

In [77]:
#run merge_data function 
cp_dep_merge = merge_data(
    crime_pop_merge, 'Crime & Population Data', 
    dep_clean, 'Deprivation Data', 
    merge_type='left', 
    left_key= 'LSOA code', 
    right_key='LSOA code (2021)', 
    force_col = 'Force'
)

In [ ]:
#check filled in averages make sense 
un_cp_dep_merge = cp_dep_merge[cp_dep_merge['LSOA code'] == 'Unknown']
m_cp_dep_merge = cp_dep_merge[cp_dep_merge['Force'] == 'Metropolitan Police'] 
m_cp_dep_merge
#filled in avgs = 25.056118	4.353288	0.28915	0.134936	15.461873	-0.103459	0.403488	27.767585	36.041049

print(m_cp_dep_merge['Index of Multiple Deprivation (IMD) Score'].describe())

A left merge was conducted and right tables null values (deprivation measures) were filled with average values per police force. Unable to calculate avergages on LSOA code or name due to null values in the crime data. This is the preferred method over losing records in an inner merge to avoid skewing aggregated crime counts.

**1c. Merge cp_dep_merge & hos_clean**

The merge uses local authority code columns and year columns as join keys creating a many-to-one relationship. e.g. Many cp_dep_merge can refer to one hos_clean record.

In [118]:
#check for unique LSOA codes for each record, numbers should be the same if true 
hos_clean['Local authority code'].nunique() #318 uniques codes 
len(hos_clean) #954 records 

hos_clean[hos_clean['Local authority code'] == 'E06000005'] #x3 duplicate records (1 for each year)

In [144]:
#check for join keys 
with pd.option_context('display.max_columns', None):
    display(cp_dep_merge.head(3), hos_clean.head(3))
    
#left - Year, Local Authority District code (2024)
#right - Year, Local authority code

In [103]:
#check which join type needs to be performed 
cp_dep_merge[cp_dep_merge['Local Authority District code (2024)'] == 'Unknown'] 
#27047 rows with unknown local authority codes - left merge 

In [154]:
#run merge_data function 
cpd_hos_merge = merge_data(
    cp_dep_merge, 'Crime, Population & Deprivation Data', 
    hos_clean, 'Housing Data', 
    merge_type='left', 
    left_key= ['Year','Local Authority District code (2024)'], 
    right_key= ['Year','Local authority code'], 
    force_col = 'Force'
) 

In [ ]:
#check filled in averages make sense 
un_cpd_hos_merge = cpd_hos_merge[cpd_hos_merge['Local Authority District code (2024)'] == 'Unknown']
m_cpd_hos_merge = cpd_hos_merge[cpd_hos_merge['Force'] == 'Metropolitan Police'] #filled in avg = 595356.605034
m_cpd_hos_merge 

#print(m_cpd_hos_merge['Median house price'].describe()) #max = 1.29m, min = 117,500, mean = 595,356

**1d. Merge cpd_hos_merge & inc_clean**

The merge uses local authority code columns and year columns as join keys creating a many-to-one relationship. e.g. Many cpd_hos_merge can refer to one inc_clean record.

In [ ]:
#check for unique LSOA codes for each record, numbers should be the same if true 
inc_clean['Local authority code'].nunique() #318 uniques codes 
len(inc_clean) #954 records 

inc_clean[inc_clean['Local authority code'] == 'E06000001'] #x3 duplicate records (1 for each year)

In [164]:
#check for join keys 
with pd.option_context('display.max_columns', None):
    display(cpd_hos_merge.head(3), inc_clean.head(3))
    
#left - Year, Local Authority District code (2024)
#right - Year, Local authority code

In [168]:
#check which join type needs to be performed 
cpd_hos_merge[cpd_hos_merge['Local Authority District code (2024)'] == 'Unknown'] 
#27047 rows with unknown local authority codes - left merge 

In [200]:
#run merge_data function 
cpdh_inc_merge = merge_data(
    cpd_hos_merge, 'Crime, Population, Deprivation & Housing Data', 
    inc_clean, 'Income Data', 
    merge_type='left', 
    left_key= ['Year','Local Authority District code (2024)'], 
    right_key= ['Year','Local authority code'], 
    force_col = 'Force'
)

In [ ]:
#check filled in averages make sense 
#un_cpd_hos_merge = cpd_hos_merge[cpd_hos_merge['Local Authority District code (2024)'] == 'Unknown']
m_cpdh_inc_merge = cpdh_inc_merge[cpdh_inc_merge['Force'] == 'Metropolitan Police'] #filled in avg =  43617.338126
m_cpdh_inc_merge

print(m_cpdh_inc_merge['Median workplace earnings'].describe()) #max =68,663 , min =26,175 , mean =43,617 

**2. Post merge cleaning**

• Drop redundant columns   
• Reassign variable name of fully merged dataframes  
• Check for duplicate records 

In [ ]:
#drop redundant columns 


In [ ]:
#reassign variable name of fully merged dataframes
crime_en_data = m_cpdh_inc_merge 

with pd.option_context('display.max_columns', None):
    display(crime_en_data.head(5))

In [ ]:
#check for duplicate records 

### Aggregation for Reporting


**Outline:** 

• 

In [ ]:
# Grain level - check for duplicates at reporting grain before aggregation
print(f"Duplicate rows at reporting grain: {crime_raw.duplicated(subset=['Falls within', 'Month', 'Crime type']).sum()}")

In [ ]:
#aggregate final crime outcome by mode 

### Export & Final Validation


**Outline:** 

• 

**3. Data validation check**

The number of rows will be printed after cleaning and compared with raw data row count. There will also be a duplicate check at the intended reporting grain. 

### Data Dictionary  

*insert data dictionary 

"   • Index of Multiple Deprivation (IMD) Score  - summarises overall deprivation in an area
   • Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs) - easier interpretation of deprivation
   • Income Score (rate) - measures the proportion of the local population experiencing income deprivation
   • Employment Score (rate) - measures the proportion of the local population involuntary excluded from the labour market
   • Education, Skills and Training Score - measures the lack of educational attainment and skills within a local population
   • 
"

### Conclusion 

brief conclusion

### References 

Office for National Statistics (2025) - Population estimates for the UK, England, Wales, Scotland and Northern Ireland: mid-2024 (https://www.ons.gov.uk/peoplepopulationandcommunity/populationandmigration/populationestimates/bulletins/annualmidyearpopulationestimates/mid2024)